<a href="https://colab.research.google.com/github/e3la/i2dc/blob/main/srt_mp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to i2dc: Make SRT for a Video - Demo Code

This is a standalone tool, meaning it doesn't need files from other parts of the i2dc toolkit to run.

### The steps are:
1.  **Check if you have a GPU:** This helps the process run faster.
2.  **Install necessary software:**
    *   **OpenAI's Whisper:** An AI for turning audio into subtitles.
    *   **FFmpeg:** An open-source video suite that turns the video into audio for Whisper.
3.  **Upload your file:** You can upload your video either directly or from a folder in your Google Drive titled `i2dc-vid`.
4.  **Run FFmpeg:** This creates an MP3 of the audio from your video file.
5.  **Give that to Whisper:** Whisper processes the audio and generates your subtitles.
6.  **Get your SRT file:** The final SRT file is produced and displayed at the end of the last cell. If you check out the file folder to the left (under the key symbol), you can see all the subtitle files that have been made.

In [ ]:
# Check if GPU is available
!nvidia-smi

In [ ]:
# Install OpenAI's Whisper
!pip install -q git+https://github.com/openai/whisper.git

In [ ]:
!sudo apt update && sudo apt install ffmpeg


In [ ]:
#@title <h1> **Upload Your Video**

# @markdown When you run this cell it will ask you to provide either the video file you want the subtitles for directly or you can connect to your google drive and choose from a list of files you put in a folder names /i2dc-vid - google drive is recommended because it is much faster.

from google.colab import files, drive
import os

video_file_path = None

while True:
    print("How would you like to provide the video file?")
    print("  1: Direct Upload from computer")
    print("  2: Select from Google Drive folder ('i2dc-vid')")
    choice = input("Enter your choice (1 or 2): ")

    if choice == '1':
        print("\nPlease select a file to upload from your computer.")
        try:
            uploaded = files.upload()
            if uploaded:
                fn = list(uploaded.keys())[0]
                video_file_path = fn
                print(f'\nUser uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
                break
            else:
                print("Upload cancelled.")
                break
        except Exception as e:
            print(f"An error occurred during upload: {e}")
            break

    elif choice == '2':
        print("\nConnecting to Google Drive...")
        drive.mount('/content/drive')

        folder_path = '/content/drive/My Drive/i2dc-vid'

        if os.path.isdir(folder_path):
            print(f"Successfully found the '{folder_path}' folder.")
            files_in_directory = os.listdir(folder_path)
            video_files = [f for f in files_in_directory if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv'))]

            if not video_files:
                print("\nNo video files found in the 'i2dc-vid' folder.")
                print("Please upload your videos to this folder in your Google Drive and run this cell again.")
                break
            else:
                print("\nPlease select a video file to process:")
                for i, video_file in enumerate(video_files):
                    print(f"  {i+1}: {video_file}")

                while True:
                    try:
                        selection = int(input("\nEnter the number of the file: "))
                        if 1 <= selection <= len(video_files):
                            selected_file = video_files[selection-1]
                            video_file_path = os.path.join(folder_path, selected_file)
                            break
                        else:
                            print(f"Invalid selection. Please enter a number between 1 and {len(video_files)}.")
                    except ValueError:
                        print("Invalid input. Please enter a number.")
                break # Exit the main while loop once a file is selected
        else:
            print("\nThe 'i2dc-vid' folder was not found in your Google Drive.")
            print("Please create a folder named 'i2dc-vid' in the main directory of your Google Drive, upload your videos, and run this cell again.")
            break
    else:
        print("\nInvalid choice. Please enter 1 or 2.")
        # The loop will continue, asking for the choice again.

if video_file_path:
    print(f"\nProcessing '{video_file_path}'...")
    # You can now use the 'video_file_path' variable for your video processing code.
else:
    print("\nNo file was selected. Please run the cell again to choose a file.")

In [ ]:
# Run FFmpeg to extract audio
!ffmpeg -i "{str(video_file_path)}" -y -q:a 0 -map a audio.mp3

In [ ]:
#i//whisper "audio.mp3" --model tiny


!whisper "audio.mp3" --model medium --language en   # For audio in english.

In [ ]:
!ls -l audio.srt
!cat audio.srt